In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter
import forestplot as fp
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
import os
from pickle import dump
from abess import LogisticRegression
from contextlib import contextmanager
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
RANDOM_STATE = 42

In [ ]:
final_clusters = pd.read_csv('../data/patient_clusters.csv', index_col=0).T
clusters = pd.DataFrame([])
for i, col in enumerate(final_clusters):
    clusters = pd.concat([clusters, pd.DataFrame(i, index=final_clusters[col].dropna(), columns=["Cluster"])])
clusters

In [ ]:
clinical_data = pd.read_csv('../data/TCGA/omics_data/raw/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)
clinical_data = clinical_data.set_index('Patient ID')
clinical_data = clinical_data[~clinical_data.index.duplicated(keep=False)]
clinical_data = pd.concat([clinical_data, clusters], join='inner', axis=1)

clinical_data['Overall Survival Status'] = clinical_data['Overall Survival Status'].str.split(':').str[0].astype(int)
survival_data = clinical_data[['Overall Survival Status', 'Overall Survival (Months)']]
recurrence_data = clinical_data[['Disease Free Status', 'Disease Free (Months)']].dropna()
recurrence_data['Disease Free Status'] = recurrence_data['Disease Free Status'].str.split(':').str[0].astype(int)

# Minimal biomarker panel

From the previous results, there are no differences in survival. However, the clusters are always very consistent, which could indicate that there is some underlying information that might be helpful or an indicator for pancreatic cancer. For that reason, we will check for biomarkers using a cross-validation and feature selection approach. Four methods will be used.

### ABESS analysis

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
months = [6, 12, 18, 24, 36, 60]

In [ ]:
@contextmanager
def fixed_seed(seed):
    state = np.random.get_state()
    np.random.seed(seed)
    try:
        yield
    finally:
        np.random.set_state(state)

class ABESSClassif(LogisticRegression):
    def __init__(self, p=None, random_state=None):
        self.p = p
        self.random_state = random_state
        with fixed_seed(self.random_state):
            super().__init__(support_size=p)

    def fit(self, X, y=None):
        with fixed_seed(self.random_state):
            super().fit(X=X, y=y)
        coef = self.coef_
        if coef.ndim == 1:
            coef = coef[np.newaxis, :]
        features = X.columns[np.where(np.any(coef != 0, axis=0))[0]]
        self.features_ = list(set(features))
        if self.p is not None:
            assert self.p == len(self.features_)
        return self

    def transform(self, X):
        return X[self.features_]

class MultiModalIFABESS(BaseEstimator, TransformerMixin):
    def __init__(self, random_state=None):
        self.random_state = random_state

    def fit(self, Xs, y):
        self.models_per_view_ = []
        self.selected_columns_per_view_ = []
        for X in Xs:
            model = ABESSClassif(random_state=self.random_state)
            model.fit(X, y)
            coef = model.coef_.ravel()
            support = np.where(coef != 0)[0]
            selected_cols = X.columns[support].tolist()
            self.models_per_view_.append(model)
            self.selected_columns_per_view_.append(selected_cols)
        selected_Xs = []
        for X, cols in zip(Xs, self.selected_columns_per_view_):
            selected_Xs.append(X[cols])
        X_combined = pd.concat(selected_Xs, axis=1)
        self.final_model_ = ABESSClassif(random_state=self.random_state)
        self.final_model_.fit(X_combined, y)
        coef_final = self.final_model_.coef_.ravel()
        mask = np.where(coef_final != 0)[0]
        self.features_ = X_combined.columns[mask].tolist()
        return self

    def transform(self, Xs):
        selected_Xs = []
        for X, cols in zip(Xs, self.selected_columns_per_view_):
            selected_Xs.append(X[cols])
        X_combined = pd.concat(selected_Xs, axis=1)
        return X_combined[self.features_]


class SplitMultiModalWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, selector, cna_cols, methyl_cols):
        self.selector = selector
        self.cna_cols = cna_cols
        self.methyl_cols = methyl_cols

    def fit(self, X, y=None):
        X_cna = X[self.cna_cols]
        X_methyl = X[self.methyl_cols]
        X_list = [X_cna, X_methyl]
        self.selector.fit(X_list, y)
        return self

    def transform(self, X):
        X_cna = X[self.cna_cols]
        X_methyl = X[self.methyl_cols]
        X_list = [X_cna, X_methyl]
        return self.selector.transform(X_list)

In [ ]:
# METHYLATION FEATURE SELECTION
methylation_data = pd.read_csv('../data/TCGA/omics_data/preprocessed/all_patients/TCGA_PDAC_Methylation.csv', index_col=0)
methylation_data = pd.concat([methylation_data, clusters], join='inner', axis=1)
X_methyl = methylation_data.drop(columns='Cluster')
y_methyl = methylation_data['Cluster']

# Matthews coefficient (all features)
crossval_methyl = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'),
                                  X_methyl, y_methyl, cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.4f, standard deviation: %0.4f" % (crossval_methyl.mean(), crossval_methyl.std()))

# Do ABESS on methylation features
pipeline_methyl = make_pipeline(ABESSClassif(random_state=RANDOM_STATE),
                                RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', oob_score=True))
scores_methyl = cross_val_score(pipeline_methyl, X_methyl, y_methyl, cv=skf, scoring='matthews_corrcoef')
pipeline_methyl.fit(X_methyl, y_methyl)
features_selected_methyl = pipeline_methyl.named_steps['abessclassif'].features_
print(f"Selected methylation features: {features_selected_methyl}")
print(f"Matthews Correlation Coefficient: {scores_methyl.mean():.4f}, standard deviation: {scores_methyl.std():.4f}")
print(f"Out-of-bag (OOB) score: {pipeline_methyl.named_steps['randomforestclassifier'].oob_score_:.4f}")

features_selected_methyl = pipeline_methyl.named_steps['abessclassif'].features_
selected_methyl_data = X_methyl[features_selected_methyl]

# SURVIVAL PLOTS
cox_methyl_survival = selected_methyl_data.merge(survival_data, left_index=True, right_index=True)
print("PLOTS (SURVIVAL):")
fig, axes = plt.subplots(1, len(features_selected_methyl), figsize=(6 * len(features_selected_methyl), 2))
df_surv = pd.DataFrame([])
print("Overall:")
for i, col_idx in enumerate(features_selected_methyl):
    biomarker = col_idx
    biomarker_data = cox_methyl_survival[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
    df_surv = pd.concat([df_surv, summary])
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_methyl) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_methyl_survival.copy()
    time_subset = cox_copy['Overall Survival (Months)'] <= time
    cox_copy.loc[~time_subset, 'Overall Survival Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_methyl), figsize=(6 * len(features_selected_methyl), 2))
    for i, col_idx in enumerate(features_selected_methyl):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
        df_surv = pd.concat([df_surv, summary])
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_methyl) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

# RECURRENCE PLOTS
cox_methyl_recurrence = selected_methyl_data.merge(recurrence_data, left_index=True, right_index=True)
print("PLOTS (RECURRENCE):")
fig, axes = plt.subplots(1, len(features_selected_methyl), figsize=(6 * len(features_selected_methyl), 2))
df_rec = pd.DataFrame([])
print("Overall:")
for i, col_idx in enumerate(features_selected_methyl):
    biomarker = col_idx
    biomarker_data = cox_methyl_recurrence[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    df_rec = pd.concat([df_rec, summary])
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_methyl) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_methyl_recurrence.copy()
    time_subset = cox_copy['Disease Free (Months)'] <= time
    cox_copy.loc[~time_subset, 'Disease Free Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_methyl), figsize=(6 * len(features_selected_methyl), 2))
    for i, col_idx in enumerate(features_selected_methyl):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
        df_rec = pd.concat([df_rec, summary])
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_methyl) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

In [ ]:
periods =  len(df_surv.index.unique())
periods = ["Overall"]*periods + ["6 months"]*periods + ["1 year"]*periods + ["18 months"]*periods + ["2 years"]*periods + ["3 years"]*periods + ["5 years"]*periods
indexes = np.unique(periods, return_index=True)[1]

In [ ]:
df_surv = df_surv.round(2)[["exp(coef)", "p"]]
df_surv["Time"] = periods
df_surv = df_surv.reset_index()
df_surv["HR(p-val)"] = df_surv["exp(coef)"].astype(str) + " (" + df_surv["p"].astype(str) + ")"
df_surv = df_surv.pivot(index="Time", columns='covariate', values='HR(p-val)')
df_surv = df_surv.loc[[periods[index] for index in sorted(indexes)]]
df_surv

In [ ]:
df_rec = df_rec.round(2)[["exp(coef)", "p"]]
df_rec["Time"] = periods
df_rec = df_rec.reset_index()
df_rec["HR(p-val)"] = df_rec["exp(coef)"].astype(str) + " (" + df_rec["p"].astype(str) + ")"
df_rec = df_rec.pivot(index="Time", columns='covariate', values='HR(p-val)')
df_rec = df_rec.loc[[periods[index] for index in sorted(indexes)]]
df_rec

In [ ]:
# COPY NUMBER SURVIVAL FEATURES
cna_data = pd.read_csv('../data/TCGA/omics_data/preprocessed/all_patients/TCGA_PDAC_CNA.csv', index_col=0)
cna_data = pd.concat([cna_data, clusters], join='inner', axis=1)
X_cna = cna_data.drop(columns='Cluster')
y_cna = cna_data['Cluster'].astype(int)

# Matthews coefficient (all features)
crossval_cna = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'),
                               X_cna, y_cna, cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.4f, standard deviation: %0.4f" % (crossval_cna.mean(), crossval_cna.std()))

# Do ABESS on copy number features
pipeline_cna = make_pipeline(ABESSClassif(random_state=RANDOM_STATE),
                             RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', oob_score=True))
scores_cna = cross_val_score(pipeline_cna, X_cna, y_cna, cv=skf, scoring='matthews_corrcoef')
pipeline_cna.fit(X_cna, y_cna)
features_selected_cna = pipeline_cna.named_steps['abessclassif'].features_
print(f"Selected copy number features: {features_selected_cna}")
print(f"Matthews Correlation Coefficient: {scores_cna.mean():.4f}, standard deviation: {scores_cna.std():.4f}")
print(f"Out-of-bag (OOB) score: {pipeline_cna.named_steps['randomforestclassifier'].oob_score_:.4f}")

features_selected_cna = pipeline_cna.named_steps['abessclassif'].features_
selected_cna_data = X_cna[features_selected_cna]

# Save model output
model = pipeline_cna.named_steps['randomforestclassifier']
save_dir = "../models" # check path directory is okay
os.makedirs(save_dir, exist_ok=True)
file_path = os.path.join(save_dir, "cna_rfmodel.pkl")
with open(file_path, "wb") as f:
    dump(model, f, protocol=5)

# SURVIVAL PLOTS
print("PLOTS (SURVIVAL):")
cox_cna_survival = selected_cna_data.merge(survival_data, left_index=True, right_index=True)
df_surv = pd.DataFrame([])
print("Overall:")
fig, axes = plt.subplots(1, len(features_selected_cna), figsize=(6 * len(features_selected_cna), 2))
for i, col_idx in enumerate(features_selected_cna):
    biomarker = col_idx
    biomarker_data = cox_cna_survival[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
    df_surv = pd.concat([df_surv, summary])
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_cna) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_cna_survival.copy()
    time_subset = cox_copy['Overall Survival (Months)'] <= time
    cox_copy.loc[~time_subset, 'Overall Survival Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_cna), figsize=(6 * len(features_selected_cna), 2))
    for i, col_idx in enumerate(features_selected_cna):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
        df_surv = pd.concat([df_surv, summary])
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_cna) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

# RECURRENCE PLOTS
print("PLOTS (RECURRENCE):")
cox_cna_recurrence = selected_cna_data.merge(recurrence_data, left_index=True, right_index=True)
fig, axes = plt.subplots(1, len(features_selected_cna), figsize=(6 * len(features_selected_cna), 2))
df_rec = pd.DataFrame([])
print("Overall:")
for i, col_idx in enumerate(features_selected_cna):
    biomarker = col_idx
    biomarker_data = cox_cna_recurrence[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
    df_rec = pd.concat([df_rec, summary])
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_cna) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_cna_recurrence.copy()
    time_subset = cox_copy['Disease Free (Months)'] <= time
    cox_copy.loc[~time_subset, 'Disease Free Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_cna), figsize=(6 * len(features_selected_cna), 2))
    for i, col_idx in enumerate(features_selected_cna):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
        df_rec = pd.concat([df_rec, summary])
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_cna) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

In [ ]:
periods =  len(df_surv.index.unique())
periods = ["Overall"]*periods + ["6 months"]*periods + ["1 year"]*periods + ["18 months"]*periods + ["2 years"]*periods + ["3 years"]*periods + ["5 years"]*periods
indexes = np.unique(periods, return_index=True)[1]

In [ ]:
df_surv = df_surv.round(2)[["exp(coef)", "p"]]
df_surv["Time"] = periods
df_surv = df_surv.reset_index()
df_surv["HR(p-val)"] = df_surv["exp(coef)"].astype(str) + " (" + df_surv["p"].astype(str) + ")"
df_surv = df_surv.pivot(index="Time", columns='covariate', values='HR(p-val)')
df_surv = df_surv.loc[[periods[index] for index in sorted(indexes)]]
df_surv

In [ ]:
df_rec = df_rec.round(2)[["exp(coef)", "p"]]
df_rec["Time"] = periods
df_rec = df_rec.reset_index()
df_rec["HR(p-val)"] = df_rec["exp(coef)"].astype(str) + " (" + df_rec["p"].astype(str) + ")"
df_rec = df_rec.pivot(index="Time", columns='covariate', values='HR(p-val)')
df_rec = df_rec.loc[[periods[index] for index in sorted(indexes)]]
df_rec

In [ ]:
# ABESS WITH INTERMEDIATE FEATURE SELECTION
X_cna = pd.read_csv("../data/TCGA/omics_data/preprocessed/all_patients/TCGA_PDAC_CNA.csv", index_col=0)
X_methylation = pd.read_csv("../data/TCGA/omics_data/preprocessed/all_patients/TCGA_PDAC_Methylation.csv", index_col=0)
combined_data = pd.concat([X_methylation, X_cna, clusters], join='inner', axis=1)
X_combined = combined_data.drop(columns='Cluster')
y_total = combined_data['Cluster']

# Matthews coefficient (all features)
crossval_total = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'),
                                 X_combined, y_total, cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.4f, standard deviation: %0.4f" % (crossval_total.mean(), crossval_total.std()))

selector = MultiModalIFABESS(random_state = RANDOM_STATE)
wrapped_selector = SplitMultiModalWrapper(selector, X_cna.columns, X_methylation.columns)
pipeline_multi = make_pipeline(wrapped_selector,
                               RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', oob_score=True))
scores_multi = cross_val_score(pipeline_multi, X_combined, y_total, cv=skf, scoring='matthews_corrcoef')
pipeline_multi.fit(X_combined, y_total)
features_selected_final = pipeline_multi.named_steps['splitmultimodalwrapper'].selector.features_
print(f"Selected multimodal features: {features_selected_final}")
print(f"Matthews Correlation Coefficient: {scores_multi.mean():.4f}, standard deviation: {scores_multi.std():.4f}")
print(f"Out-of-bag (OOB) score: {pipeline_multi.named_steps['randomforestclassifier'].oob_score_:.4f}")

# SURVIVAL PLOTS
selected_final_data = X_combined[features_selected_final]
cox_input_survival = selected_final_data.merge(survival_data, left_index=True, right_index=True)
print("PLOTS (SURVIVAL):")
df_surv = pd.DataFrame([])
print("Overall:")
fig, axes = plt.subplots(1, len(features_selected_final), figsize=(6 * len(features_selected_final), 2))
for i, col_idx in enumerate(features_selected_final):
    biomarker = col_idx
    biomarker_data = cox_input_survival[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
    df_surv = pd.concat([df_surv, summary])
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_final) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_input_survival.copy()
    time_subset = cox_copy['Overall Survival (Months)'] <= time
    cox_copy.loc[~time_subset, 'Overall Survival Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_final), figsize=(6 * len(features_selected_final), 2))
    for i, col_idx in enumerate(features_selected_final):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
        df_surv = pd.concat([df_surv, summary])
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_final) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

# RECURRENCE PLOTS
cox_input_recurrence = selected_final_data.merge(recurrence_data, left_index=True, right_index=True)
print("PLOTS (RECURRENCE):")
df_rec = pd.DataFrame([])
print("Overall:")
fig, axes = plt.subplots(1, len(features_selected_final), figsize=(6 * len(features_selected_final), 2))
for i, col_idx in enumerate(features_selected_final):
    biomarker = col_idx
    biomarker_data = cox_input_recurrence[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
    df_rec = pd.concat([df_rec, summary])
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_final) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_input_recurrence.copy()
    time_subset = cox_copy['Disease Free (Months)'] <= time
    cox_copy.loc[~time_subset, 'Disease Free Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_final), figsize=(6 * len(features_selected_final), 2))
    for i, col_idx in enumerate(features_selected_final):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
        df_rec = pd.concat([df_rec, summary])
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_final) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

In [ ]:
periods =  len(df_surv.index.unique())
periods = ["Overall"]*periods + ["6 months"]*periods + ["1 year"]*periods + ["18 months"]*periods + ["2 years"]*periods + ["3 years"]*periods + ["5 years"]*periods
indexes = np.unique(periods, return_index=True)[1]

In [ ]:
df_surv = df_surv.round(2)[["exp(coef)", "p"]]
df_surv["Time"] = periods
df_surv = df_surv.reset_index()
df_surv["HR(p-val)"] = df_surv["exp(coef)"].astype(str) + " (" + df_surv["p"].astype(str) + ")"
df_surv = df_surv.pivot(index="Time", columns='covariate', values='HR(p-val)')
df_surv = df_surv.loc[[periods[index] for index in sorted(indexes)]]
df_surv

In [ ]:
df_rec = df_rec.round(2)[["exp(coef)", "p"]]
df_rec["Time"] = periods
df_rec = df_rec.reset_index()
df_rec["HR(p-val)"] = df_rec["exp(coef)"].astype(str) + " (" + df_rec["p"].astype(str) + ")"
df_rec = df_rec.pivot(index="Time", columns='covariate', values='HR(p-val)')
df_rec = df_rec.loc[[periods[index] for index in sorted(indexes)]]
df_rec